# Kwant cross-check of the effective-mass NEGF solver

Independent confirmation for **Sec. IV-A** of *Verified Differentiable NEGF for
Device Inverse Design*: the same double-barrier structure solved by
[Kwant](https://kwant-project.org), which uses wavefunction mode-matching
rather than Green's functions, agrees with this solver to **1.17e-12**.

Kwant has no Windows pip wheel, so `verify/check_claims.py` reports this step
as SKIP on Windows. This notebook runs it on Linux in a few minutes.

**Run order:** execute cell 1, let the runtime restart, then run the rest.
The restart is required because Kwant needs NumPy 1.x while Colab ships 2.x.


## 1. Install dependencies (triggers a runtime restart)

In [ ]:
# Kwant needs NumPy 1.x. SciPy >= 1.14 drags NumPy 2 back in and the Kwant
# C extension then fails to compile ("PyArray_Descr has no member 'subarray'"),
# so scipy must be pinned alongside numpy.
!pip install -q "numpy<2" "scipy<1.14" cython tinyarray setuptools wheel
!pip install -q --no-build-isolation kwant

import numpy, os
if numpy.__version__.startswith("2"):
    print(f"numpy {numpy.__version__} still active -- restarting runtime.")
    print("Re-run from cell 2 once the runtime comes back.")
    os.kill(os.getpid(), 9)
else:
    print(f"numpy {numpy.__version__} -- no restart needed.")

## 2. Clone the solver at a pinned revision

In [ ]:
# Pin to the archived release so this notebook reproduces the published
# number. Use "main" only for development.
REV = "main"   # e.g. "v1.0.0" once the release is cut

!git clone -q https://github.com/QuaNaD-Lab-PESU/diff-negf.git /content/diff-negf
%cd /content/diff-negf
!git checkout -q $REV
!pip install -q .
!git log --oneline -1

## 3. Confirm the environment

In [ ]:
import kwant, numpy
print("kwant", kwant.__version__, "| numpy", numpy.__version__)
assert numpy.__version__.startswith("1"), "NumPy 1.x required -- re-run cell 1"

## 4. Run the cross-check

In [ ]:
!python /content/diff-negf/verification/kwant_crosscheck.py

## Expected result

The script prints the deviation it measures and its own PASS/FAIL verdict. The
manuscript reports a maximum deviation of **1.17e-12** between the NEGF solver
and Kwant on the double-barrier structure of Fig. 3(b).

This residual is roundoff-limited and depends on the host linear-algebra
build, so the value here will not match the paper digit for digit. An
independent re-run on Linux (Kwant 1.5.0, NumPy 1.26.4, SciPy 1.13.1, SciPy
fallback solver rather than MUMPS) gave **4.922e-13** -- same order, and the
script reported PASS. Kwant on Colab likewise falls back from MUMPS, so expect
something near 5e-13 here.

Agreement at the 1e-12--1e-13 level confirms the cross-check; it lies ten
orders of magnitude below any physically interesting feature of the
transmission. A deviation orders of magnitude *larger* would indicate a
genuine discrepancy.